## 27 — Citation Network Analysis

Two complementary directions:

**Option B — Who Cites Award Papers? (forward citations)**
- Plot 1: Year distribution of citing papers — when does impact peak?
- Plot 2: Field distribution — are award papers cited mostly within CS or cross-disciplinary?
- Plot 3: Connector papers (`cites_n_award_papers >= 2`) — count distribution + field breakdown
- Plot 4: Citation lag heatmap — for each award cohort year, when did citations peak?

**Option C — Referenced (nb-24) vs. Citing (nb-26): what award papers built on vs. what built on them**
- Plot 5: Field comparison side-by-side (cited vs. citing)
- Plot 6: Publication year KDE — backward vs. forward temporal footprint
- Plot 7: Source type comparison (journal / conference / repository)
- Plot 8: Topic shift heatmap — award paper field → field of citers

**Inputs:**
- `data/matched/citing_papers.csv`        — 438,807 papers that cite award papers
- `data/matched/award_to_citing_edges.csv` — 507,718 award→citer edges
- `data/matched/cited_papers.csv`         — 26,921 papers cited by award papers
- `data/matched/award_to_cited_edges.csv` — 34,417 award→cited edges
- `data/matched/huang_matched_openalex.csv` — 890 award papers

**Output figures:** saved to `figures/` as `p27_0X_*.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.stats import gaussian_kde

plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'axes.edgecolor'   : 'black',
    'axes.linewidth'   : 0.8,
    'axes.grid'        : False,
    'font.family'      : 'Arial',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'legend.frameon'   : True,
    'legend.edgecolor' : '#cccccc',
})

HATCHES = ['///', 'xxx', '...', '\\\\', '+++', '---', '|||']
GREYS   = ['black', '#444', '#777', '#999', '#bbb', '#ddd']

DATA = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched')
FIG  = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\figures')
FIG.mkdir(parents=True, exist_ok=True)

citing  = pd.read_csv(DATA / 'citing_papers.csv', low_memory=False)
c_edges = pd.read_csv(DATA / 'award_to_citing_edges.csv')
cited   = pd.read_csv(DATA / 'cited_papers.csv',  low_memory=False)
r_edges = pd.read_csv(DATA / 'award_to_cited_edges.csv')
awards  = pd.read_csv(DATA / 'huang_matched_openalex.csv')

print(f'citing_papers        : {len(citing):,} rows')
print(f'award_to_citing_edges: {len(c_edges):,} rows')
print(f'cited_papers         : {len(cited):,} rows')
print(f'award_to_cited_edges : {len(r_edges):,} rows')
print(f'award_papers         : {len(awards):,} rows')

---
## Option B — Who Cites Award Papers?
### Plot 1 — Year distribution of citing papers

In [ ]:
yr = citing['publication_year'].dropna().astype(int)
yr_counts = yr[(yr >= 2000) & (yr <= 2025)].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yr_counts.index, yr_counts.values,
       color='black', edgecolor='black', linewidth=0.4, width=0.8)
ax.set_xlabel('Publication Year')
ax.set_ylabel('Number of Citing Papers')
ax.set_title('Publication Year Distribution of Papers Citing Award Papers (2000–2025)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig(FIG / 'p27_01_citing_year_dist.png', dpi=150)
plt.show()
print(f'Peak year: {yr_counts.idxmax()} ({yr_counts.max():,} papers)')

### Plot 2 — Field distribution of citing papers

In [ ]:
field_counts = (
    citing['top_field']
    .replace('', pd.NA)
    .dropna()
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(field_counts.index[::-1], field_counts.values[::-1],
               color='black', edgecolor='black', linewidth=0.4)
ax.set_xlabel('Number of Citing Papers')
ax.set_title('Top 15 Fields of Papers Citing Award Papers')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(FIG / 'p27_02_citing_field_dist.png', dpi=150)
plt.show()

pct_cs = field_counts.get('Computer Science', 0) / field_counts.sum() * 100
print(f'CS share of top-15 fields: {pct_cs:.1f}%')

### Plot 3 — Connector papers (cite ≥ 2 award papers)

In [ ]:
citing['cites_n_award_papers'] = pd.to_numeric(
    citing['cites_n_award_papers'], errors='coerce'
).fillna(1).astype(int)

# ---- 3a: distribution of cites_n_award_papers (1-10+)
n_dist = citing['cites_n_award_papers'].clip(upper=10).value_counts().sort_index()
labels = [str(i) if i < 10 else '10+' for i in n_dist.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, n_dist.values, color='black', edgecolor='black', linewidth=0.4)
axes[0].set_xlabel('Number of Award Papers Cited')
axes[0].set_ylabel('Number of Citing Papers')
axes[0].set_title('How Many Award Papers Does Each Citer Cite?')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ---- 3b: field breakdown for connectors (≥2)
connectors = citing[citing['cites_n_award_papers'] >= 2]
conn_fields = (
    connectors['top_field']
    .replace('', pd.NA)
    .dropna()
    .value_counts()
    .head(10)
)

axes[1].barh(conn_fields.index[::-1], conn_fields.values[::-1],
             color='#444', edgecolor='black', linewidth=0.4)
axes[1].set_xlabel('Number of Connector Papers')
axes[1].set_title(f'Fields of Connector Papers (cite ≥2 award papers, n={len(connectors):,})')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(FIG / 'p27_03_connector_papers.png', dpi=150)
plt.show()
print(f'Connector papers (≥2): {len(connectors):,}  ({len(connectors)/len(citing)*100:.1f}% of all citers)')

### Plot 4 — Citation lag heatmap (award cohort year → citing year)

In [ ]:
# Merge edges with award year + citing paper year
c_edges['award_year'] = pd.to_numeric(c_edges['award_year'], errors='coerce')
cite_yr = citing[['openalex_id', 'publication_year']].copy()
cite_yr.columns = ['citing_paper_id', 'citing_year']

lag_df = c_edges.merge(cite_yr, on='citing_paper_id', how='left')
lag_df['citing_year'] = pd.to_numeric(lag_df['citing_year'], errors='coerce')

lag_df = lag_df[
    lag_df['award_year'].between(2000, 2018) &
    lag_df['citing_year'].between(2000, 2025)
]

pivot = (
    lag_df.groupby(['award_year', 'citing_year'])
    .size()
    .unstack(fill_value=0)
)

# Normalise each row to show relative peak (% of that cohort's total citations)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot_pct.values, aspect='auto', cmap='Greys',
               origin='lower', vmin=0, vmax=pivot_pct.values.max())

ax.set_xticks(range(len(pivot_pct.columns)))
ax.set_xticklabels(pivot_pct.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot_pct.index)))
ax.set_yticklabels(pivot_pct.index, fontsize=9)
ax.set_xlabel('Year of Citing Paper')
ax.set_ylabel('Award Paper Cohort Year')
ax.set_title('Citation Lag Heatmap\n(% of each award cohort\'s citations, by citing year)')
plt.colorbar(im, ax=ax, label='% of cohort citations')
plt.tight_layout()
plt.savefig(FIG / 'p27_04_citation_lag_heatmap.png', dpi=150)
plt.show()

---
## Option C — Referenced (backward) vs. Citing (forward)
### Plot 5 — Field distribution: cited vs. citing (side-by-side)

In [ ]:
# Top-15 union of fields across both datasets
ref_fields = cited['top_field'].replace('', pd.NA).dropna().value_counts()
cit_fields = citing['top_field'].replace('', pd.NA).dropna().value_counts()

top_fields = (
    pd.concat([ref_fields, cit_fields])
    .groupby(level=0).sum()
    .sort_values(ascending=False)
    .head(15)
    .index.tolist()
)

ref_vals = ref_fields.reindex(top_fields, fill_value=0)
cit_vals = cit_fields.reindex(top_fields, fill_value=0)

# Normalise to % so the very different total sizes are comparable
ref_pct = ref_vals / ref_vals.sum() * 100
cit_pct = cit_vals / cit_vals.sum() * 100

x = np.arange(len(top_fields))
w = 0.38

fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(x + w/2, ref_pct.values, height=w, color='black',  label='Cited (backward)', hatch='///')
ax.barh(x - w/2, cit_pct.values, height=w, color='#888',   label='Citing (forward)')
ax.set_yticks(x)
ax.set_yticklabels(top_fields, fontsize=9)
ax.set_xlabel('Share of Papers in Each Dataset (%)')
ax.set_title('Field Distribution: Papers Cited BY Award Papers vs. Papers Citing Award Papers')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'p27_05_field_cited_vs_citing.png', dpi=150)
plt.show()

### Plot 6 — Publication year KDE: backward vs. forward temporal footprint

In [ ]:
ref_yr = cited['publication_year'].dropna().astype(float)
cit_yr = citing['publication_year'].dropna().astype(float)

ref_yr = ref_yr[(ref_yr >= 1960) & (ref_yr <= 2025)]
cit_yr = cit_yr[(cit_yr >= 1960) & (cit_yr <= 2025)]

xs = np.linspace(1960, 2025, 500)
kde_ref = gaussian_kde(ref_yr, bw_method=0.15)
kde_cit = gaussian_kde(cit_yr, bw_method=0.15)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(xs, kde_ref(xs), alpha=0.45, color='black',  label='Cited (backward)')
ax.fill_between(xs, kde_cit(xs), alpha=0.35, color='#888',   label='Citing (forward)')
ax.plot(xs, kde_ref(xs), color='black',  linewidth=1)
ax.plot(xs, kde_cit(xs), color='#555',   linewidth=1, linestyle='--')
ax.set_xlabel('Publication Year')
ax.set_ylabel('Density')
ax.set_title('Temporal Footprint: Papers Cited BY Award Papers vs. Papers Citing Award Papers')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'p27_06_year_kde_cited_vs_citing.png', dpi=150)
plt.show()

### Plot 7 — Source type: cited vs. citing (journal / conference / repository)

In [ ]:
ref_src = cited['source_type'].replace('', pd.NA).dropna().value_counts(normalize=True) * 100
cit_src = citing['source_type'].replace('', pd.NA).dropna().value_counts(normalize=True) * 100

all_types = sorted(set(ref_src.index) | set(cit_src.index))
ref_src = ref_src.reindex(all_types, fill_value=0)
cit_src = cit_src.reindex(all_types, fill_value=0)

x = np.arange(len(all_types))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, ref_src.values, width=w, color='black', label='Cited (backward)', hatch='///')
ax.bar(x + w/2, cit_src.values, width=w, color='#888',  label='Citing (forward)')
ax.set_xticks(x)
ax.set_xticklabels(all_types, rotation=30, ha='right')
ax.set_ylabel('Share (%)')
ax.set_title('Source Type: Papers Cited BY Award Papers vs. Papers Citing Award Papers')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'p27_07_source_type_cited_vs_citing.png', dpi=150)
plt.show()

### Plot 8 — Topic shift heatmap: award paper field → field of citing papers

In [ ]:
# Build award_field lookup
award_field = (
    awards[['openalex_id', 'top_field']]
    .dropna(subset=['top_field'])
    .rename(columns={'openalex_id': 'award_paper_id', 'top_field': 'award_field'})
)

# Build citing_field lookup
citing_field = (
    citing[['openalex_id', 'top_field']]
    .dropna(subset=['top_field'])
    .rename(columns={'openalex_id': 'citing_paper_id', 'top_field': 'citing_field'})
)

# Join both into the edge table
shift = (
    c_edges
    .merge(award_field,   on='award_paper_id',  how='inner')
    .merge(citing_field,  on='citing_paper_id', how='inner')
)

# Keep top-10 award fields + top-10 citing fields for readability
top_award_fields  = shift['award_field'].value_counts().head(10).index.tolist()
top_citing_fields = shift['citing_field'].value_counts().head(10).index.tolist()

shift_sub = shift[
    shift['award_field'].isin(top_award_fields) &
    shift['citing_field'].isin(top_citing_fields)
]

pivot_shift = (
    shift_sub
    .groupby(['award_field', 'citing_field'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=top_award_fields, columns=top_citing_fields, fill_value=0)
)

# Normalise rows → % of that award field's citations going to each citing field
pivot_pct = pivot_shift.div(pivot_shift.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(pivot_pct.values, aspect='auto', cmap='Greys', vmin=0)

ax.set_xticks(range(len(pivot_pct.columns)))
ax.set_xticklabels(pivot_pct.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_pct.index)))
ax.set_yticklabels(pivot_pct.index, fontsize=8)
ax.set_xlabel('Field of Citing Paper')
ax.set_ylabel('Field of Award Paper')
ax.set_title('Topic Shift Heatmap\n(% of award field\'s citations received from each citing field)')
plt.colorbar(im, ax=ax, label='% of citations')

# Annotate cells
for i in range(len(pivot_pct.index)):
    for j in range(len(pivot_pct.columns)):
        v = pivot_pct.values[i, j]
        if v > 1:
            text_color = 'white' if v > pivot_pct.values.max() * 0.6 else 'black'
            ax.text(j, i, f'{v:.0f}%', ha='center', va='center',
                    fontsize=7, color=text_color)

plt.tight_layout()
plt.savefig(FIG / 'p27_08_topic_shift_heatmap.png', dpi=150)
plt.show()
print('Done — all 8 figures saved to figures/')